In [0]:
!pip install pyomo

In [0]:
from pyspark.sql import functions as F


# 1. Input data

# 1.1. Price per unit

In [0]:
date_cutoff = '2023-01-01'
material = 'ED13'
'''rent = spark.table('rentabilidad_material_silver_table').filter(
    (F.col('material') == 'ED13') &
    (F.to_date(F.col('week-month-year')) > F.lit(date_cutoff))
)'''
litros = spark.table('litros_material_silver_table')
metadata = spark.table('metadata_materiales_optimized')
ventas = spark.table('ventas_material_silver_table').filter(
    (F.col('material') == 'ED13') &
    (F.to_date(F.col('week-month-year')) > F.lit(date_cutoff))
)

In [0]:
meterial_precio = ventas.join(litros, ['week-month-year', 'establecimiento', 'material'], how='inner').withColumnRenamed("establecimiento", "idsap")
metadata = metadata.withColumn(
    "volumen",
    F.when(F.col("volumen").contains(" L"),
           F.regexp_replace(F.regexp_replace(F.col("volumen"), ",", "."), " L", "").cast("double"))
    .when(F.col("volumen").contains("DM3"),
           F.regexp_replace(F.regexp_replace(F.col("volumen"), ",", "."), " DM3", "").cast("double"))
    .when(F.col("volumen").contains("CM3"),  # ← check this BEFORE M3!
          F.regexp_replace(F.regexp_replace(F.col("volumen"), ",", "."), " CM3", "").cast("double") * 0.001)
    .when(F.col("volumen").contains("M3"),
          F.regexp_replace(F.regexp_replace(F.col("volumen"), ",", "."), " M3", "").cast("double") * 1000)
    .otherwise(None)
).select('material_id', 'volumen')

meterial_precio = meterial_precio.join(
    metadata.select("material_id", "volumen"),
    meterial_precio.material == metadata.material_id,
    how="left"
)

meterial_precio = meterial_precio.withColumn(
    "n_material",
    F.round(F.try_divide(F.col("litros"), F.col("volumen")), 2)
)

meterial_precio = meterial_precio.withColumn(
    "euro_material",
    F.round(F.try_divide(F.col("ventas_netas"), F.col("n_material")), 2)
)
print("n rows: ", meterial_precio.count())
display(meterial_precio)

## 1.2. Metadata

In [0]:
cols_to_use = ['idsap', 'rating_google', 'userratingcount_og', 'distance_center_metros', 'angulo_deg', 'od_is_coast_distance_raw']
google_delectatech_geo = spark.table('bronze_metadata_features')

In [0]:
from pyspark.sql.functions import col, count, when

def null_percentage_readable(df):
    total_rows = df.count()
    nulls = []
    for c in df.columns:
        dtype = dict(df.dtypes)[c]
        if dtype == 'string':
            # For string columns: check for nulls and empty strings
            null_count = df.filter(col(c).isNull() | (col(c) == "")).count()
        else:
            # For other types: only check for nulls
            null_count = df.filter(col(c).isNull()).count()
        nulls.append((c, (null_count / total_rows) * 100))
    return spark.createDataFrame(nulls, ["column_name", "null_percentage"])

# Example usage
null_ptg_df = null_percentage_readable(google_delectatech_geo)
display(null_ptg_df.filter(F.col("null_percentage") < 40))

In [0]:
google_delectatech_geo.count()

In [0]:
google_delectatech_geo = google_delectatech_geo.select(cols_to_use).dropna(how="any")
print("n rows: ", google_delectatech_geo.count())
display(google_delectatech_geo.limit(5))

In [0]:
merged_df = meterial_precio.join(google_delectatech_geo, ['idsap'], how='inner')
print("n rows: ", merged_df.count())
display(merged_df.limit(5))

In [0]:
display(merged_df.filter(F.col('idsap') == '8100009794'))

In [0]:
unique_df = merged_df.select('idsap').distinct()
print("n rows: ", unique_df.count())
display(unique_df)

# Training

In [0]:
import random
random.seed(42)

import torch

from trainer import Trainer

In [0]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

'''model_config = {
    'hidden_layers': 3,
    'hidden_size': 64,
    'drop': 0.2
    }

training_config = {
    'lr': 0.001,
    'wd': 0.001,
    'batchsize': 128,
    'iters': 5000,
    'patience': 100
    }'''

config = {
    'hidden_layers': 3,
    'hidden_size': 16,
    'drop': 0.35427615413058317,
    'lr': 0.007826501180523646,
    'wd': 0.016333374566295637,
    'batchsize': 352,
    'iters': 5000,
    'patience': 100
}


In [0]:
def pick_n_random_ids(df, n_samples):
    ordered_df = (
        df
        .select("idsap")
        .distinct()
        .orderBy("idsap")  # consistent sort to fix partitioning
    )

    # Step 2: add reproducible random column
    randomized_df = ordered_df.withColumn("rand", F.rand(seed=42))

    # Step 3: order by random value and select first N
    unique_ids = (
        randomized_df
        .orderBy("rand")
        .limit(n_samples)
        .select("idsap")
        .toPandas()["idsap"]
        .tolist()
    )
    return unique_ids

In [0]:
unique_ids = pick_n_random_ids(unique_df, n_samples=100)
print("n establecimientos: ", len(unique_ids))

train_data = merged_df.filter(F.col('idsap').isin(unique_ids)).dropna(how="any")
print("n rows: ", train_data.count())

In [0]:
display(train_data)

In [0]:
# Filter weird euro_material

train_data = train_data.filter(F.col('euro_material') >= 10)

In [0]:
fetures = [col for col in (cols_to_use + ['euro_material']) if col != 'idsap']
#train_data = merged_df#.filter(F.col('idsap') == '8100009794')
y = train_data.select('euro_material').toPandas()['euro_material']
X = train_data.select(fetures).toPandas()

trainer = Trainer(X, y, config, device=device, path_model='./models/best_model.pth')
trainer.create_data_partitions(val_size=0.2, test_size=0.2, random_state=42)

print("X_train shape: ", trainer.X_train.shape)
print("X_val shape: ", trainer.X_val.shape)
print("X_test shape: ", trainer.X_test.shape)

best_loss = trainer.train()

trainer.test()

In [0]:
best_loss

In [0]:
trainer.model

In [0]:
trainer.evaluate_gaussian_nll()
trainer.evaluate_standardized_residuals()

# Hyperopt

In [0]:
!pip install hyperopt==0.2.7

In [0]:
import mlflow

from hyperopt import fmin, tpe, hp, SparkTrials, Trials, STATUS_OK
from hyperopt.pyll.base import scope

import sys
sys.path.append("/Workspace/Repos/Shared/FunctionsUtils/Training")

from Utils import get_formatted_datetime

In [0]:
mlClient = mlflow.MlflowClient()

#Experimento creado a mano previamente
experiment_id = '3365612151366799'
experiment = mlClient.get_experiment(experiment_id)

print(experiment)

In [0]:
import uuid
import os

import numpy as np

In [0]:
def objective(params):
    mlflow.lightgbm.autolog()
    with mlflow.start_run(nested=True, experiment_id=experiment_id):
        mlflow.set_tag('source', 'Price_to_demand_hyperparameter_search_' + '1')

        # Log hyperparameters used in this trial
        mlflow.log_params(params)

        config = {
            'hidden_layers': params['hidden_layers'],
            'hidden_size': params['hidden_size'],
            'drop': params['drop'],
            'lr': params['lr'],
            'wd': params['wd'],
            'batchsize': params['batchsize'],
            'iters': 5000,
            'patience': params['patience'],  # still fixed, but now pulled from the search space
        }

        model_path = f"/tmp/best_model_{uuid.uuid4().hex}.pth"
        os.makedirs(os.path.dirname(model_path), exist_ok=True)

        trainer = Trainer(X, y, config, device=device, path_model=model_path)
        trainer.create_data_partitions(val_size=0.2, test_size=0.2, random_state=42)

        trainer.train()

        # Get baseline and model loss (both evaluated on validation set)
        baseline_loss, val_loss = trainer.evaluate_gaussian_nll()

        # Improvements
        absolute_improvement = baseline_loss - val_loss
        relative_improvement = absolute_improvement / baseline_loss

        # Log metrics
        mlflow.log_metric("val_nll", val_loss)
        mlflow.log_metric("baseline_nll", baseline_loss)
        mlflow.log_metric("absolute_improvement", absolute_improvement)
        mlflow.log_metric("relative_improvement", relative_improvement)

        #os.remove(model_path)

        return {
            'status': STATUS_OK,
            'loss': val_loss
        }



def hyperparams_search(custom_run_name, n_hyperparams_runs=50):
    runName = f"Price to demand - {custom_run_name} {get_formatted_datetime()}"
    with mlflow.start_run(run_name=runName, experiment_id=experiment_id, description=runName) as run:

        search_space = {
            # Model architecture
            'hidden_layers': scope.int(hp.quniform('hidden_layers', 1, 5, 1)),   # try 1–5 layers
            'hidden_size': scope.int(hp.quniform('hidden_size', 16, 256, 16)),   # try 32 to 256 in steps of 32
            'drop': hp.uniform('drop', 0.0, 0.5),                                 # dropout between 0% and 50%

            # Training parameters
            'lr': hp.loguniform('lr', -7, -3),                                   # log-scale: ~1e-7 to 1e-3
            'wd': hp.loguniform('wd', -8, -2),                                   # log-scale: ~1e-8 to 1e-2
            'batchsize': scope.int(hp.quniform('batchsize', 32, 512, 32)),       # 64–512 in steps of 64
            'patience': 100 # fixed patience
        }
    
        spark_trials = SparkTrials(parallelism=20)

        best_params_lightgbm = fmin(
            fn=objective, 
            space=search_space, 
            algo=tpe.suggest, 
            max_evals=n_hyperparams_runs,
            trials=spark_trials,
        )

In [0]:
y = train_data.select('euro_material').toPandas()['euro_material']
X = train_data.select(fetures).toPandas()
hyperparams_search("test", 100)

In [0]:
mlflow.search_runs(experiment_ids=[experiment_id], filter_string="tags.source = 'Price_to_demand_hyperparameter_search_" +  "1" + "'", order_by=['metrics.val_nll ASC']).iloc[0].run_id

In [0]:
mlflow.search_runs(experiment_ids=[experiment_id], filter_string="tags.source = 'Price_to_demand_hyperparameter_search_" +  "1" + "'", order_by=['metrics.val_nll ASC']).iloc[0]['metrics.val_nll']